In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Root:", PROJECT_ROOT)

Root: /Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction


In [3]:
ipo = pd.read_csv(DATA_RAW / "ipo_master.csv")

ipo.columns

Index(['COMPANY NAME', 'SECURITY TYPE', 'ISSUE PRICE', 'Symbol',
       'ISSUE START DATE', 'ISSUE END DATE', 'PRICE RANGE', 'DATE OF LISTING'],
      dtype='object')

In [4]:
# Normalize IPO master columns explicitly (NO guessing)

ipo = pd.read_csv(DATA_RAW / "ipo_master.csv")

ipo["listing_date"] = pd.to_datetime(
    ipo["DATE OF LISTING"],
    utc=True,
    errors="coerce"
)

ipo["company_name"] = (
    ipo["COMPANY NAME"]
    .str.lower()
    .str.strip()
)

ipo = ipo[["company_name", "listing_date"]].dropna()

ipo.shape

/var/folders/fk/0kqynd_95d71j1htw9b057j00000gn/T/ipykernel_20458/3734469814.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ipo["listing_date"] = pd.to_datetime(


(1179, 2)

In [5]:
# Load and combine NSE bhavcopy CSVs

BHAV_PATH = DATA_RAW / "nse_bhavcopy"

bhav_files = list(BHAV_PATH.glob("*.csv"))
print("Bhavcopy files found:", len(bhav_files))

bhav_list = []

for f in bhav_files:
    df = pd.read_csv(f)
    
    # normalize column names
    df.columns = [c.lower().strip() for c in df.columns]
    
    df = df[["symbol", "close", "date"]]
    df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
    
    bhav_list.append(df)

bhav = pd.concat(bhav_list, ignore_index=True)

bhav = bhav.dropna(subset=["date", "close"])

bhav.shape

Bhavcopy files found: 2108


(3367539, 3)

In [6]:
market_daily = (
    bhav
    .groupby("date", as_index=False)["close"]
    .mean()
    .rename(columns={"close": "market_close"})
    .sort_values("date")
)

market_daily.head()

,date,market_close
0,2016-01-01 00:00:00+00:00,459.224833
1,2016-01-04 00:00:00+00:00,458.036950
2,2016-01-05 00:00:00+00:00,462.153756
3,2016-01-06 00:00:00+00:00,460.753105
4,2016-01-07 00:00:00+00:00,451.017534


In [7]:
# Market returns
market_daily["market_return_1d"] = market_daily["market_close"].pct_change()
market_daily["market_return_7d"] = market_daily["market_close"].pct_change(7)
market_daily["market_return_30d"] = market_daily["market_close"].pct_change(30)

# Market volatility (rolling std of daily returns)
market_daily["market_volatility_7d"] = (
    market_daily["market_return_1d"]
    .rolling(7)
    .std()
)

market_daily["market_volatility_30d"] = (
    market_daily["market_return_1d"]
    .rolling(30)
    .std()
)

In [8]:
market_daily["market_vol_30d"] = (
    market_daily["market_return_1d"]
    .rolling(30, min_periods=15)
    .std()
)

market_daily.head(40)

,date,market_close,market_return_1d,market_return_7d,market_return_30d,market_volatility_7d,market_volatility_30d,market_vol_30d
0,2016-01-01 00:00:00+00:00,459.224833,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-01-04 00:00:00+00:00,458.036950,-0.002587,NaN,NaN,NaN,NaN,NaN
2,2016-01-05 00:00:00+00:00,462.153756,0.008988,NaN,NaN,NaN,NaN,NaN
3,2016-01-06 00:00:00+00:00,460.753105,-0.003031,NaN,NaN,NaN,NaN,NaN
4,2016-01-07 00:00:00+00:00,451.017534,-0.021130,NaN,NaN,NaN,NaN,NaN
5,2016-01-08 00:00:00+00:00,460.506020,0.021038,NaN,NaN,NaN,NaN,NaN
6,2016-01-11 00:00:00+00:00,457.567938,-0.006380,NaN,NaN,NaN,NaN,NaN
7,2016-01-12 00:00:00+00:00,453.415490,-0.009075,-0.012650,NaN,0.013464,NaN,NaN
8,2016-01-13 00:00:00+00:00,443.511492,-0.021843,-0.031712,NaN,0.015482,NaN,NaN
9,2016-01-14 00:00:00+00:00,438.970607,-0.010238,-0.050163,NaN,0.014356,NaN,NaN


In [9]:
ipo_market = pd.merge_asof(
    ipo.sort_values("listing_date"),
    market_daily.sort_values("date"),
    left_on="listing_date",
    right_on="date",
    direction="backward"
)

ipo_market.shape

(1179, 10)

In [10]:
ipo_market["bull_market"] = (
    ipo_market["market_return_30d"] > 0.05
).astype(int)

ipo_market["bear_market"] = (
    ipo_market["market_return_30d"] < -0.05
).astype(int)

ipo_market["high_volatility"] = (
    ipo_market["market_vol_30d"] >
    ipo_market["market_vol_30d"].expanding().median()
).astype(int)

ipo_market[["bull_market", "bear_market", "high_volatility"]].mean()

bull_market        0.430874
bear_market        0.112807
high_volatility    0.500424
dtype: float64

In [11]:
ipo_market["recent_market_trend"] = (
    ipo_market["market_return_7d"] -
    ipo_market["market_return_30d"]
)

ipo_market["days_from_market_start"] = (
    ipo_market["listing_date"] -
    market_daily["date"].min()
).dt.days

ipo_market[
    ["recent_market_trend", "days_from_market_start"]
].describe()

,recent_market_trend,days_from_market_start
count,1156.000000,1179.000000
mean,-0.019979,2305.338422
std,0.051736,1007.849097
min,-0.190705,-4747.000000
25%,-0.055327,1749.000000
50%,-0.031539,2666.000000
75%,0.015252,3036.000000
max,0.141440,3638.000000


In [12]:
market_features = ipo_market[
    [
        "company_name",
        "listing_date",
        "market_return_1d",
        "market_return_7d",
        "market_return_30d",
        "market_vol_30d",
        "recent_market_trend",
        "bull_market",
        "bear_market",
        "high_volatility",
        "days_from_market_start"
    ]
].dropna()

In [13]:
market_features.isna().mean()

company_name              0.0
listing_date              0.0
market_return_1d          0.0
market_return_7d          0.0
market_return_30d         0.0
market_vol_30d            0.0
recent_market_trend       0.0
bull_market               0.0
bear_market               0.0
high_volatility           0.0
days_from_market_start    0.0
dtype: float64

In [14]:
OUT_PATH = DATA_PROCESSED / "market_features.csv"

market_features.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)

Saved: /Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/processed/market_features.csv
